# Apart from returns, what other variables would make a good ML model?

1) Risk & distribution stats (from prices)

What: realized vol/semivol, downside deviation, skew/kurtosis, rolling drawdowns, rolling betas/correlations, idiosyncratic vol.

Where: compute from Ticker.history() (OHLCV). yfinance provides split/dividend-adjusted series; you compute the stats. 
ranaroussi.github.io

2) Factor exposures / factor returns

What: exposures to market, size, value, momentum, profitability, investment; later map to tracking error and factor constraints.

Where: use Kenneth French Data Library for daily/monthly factor returns (FF3/FF5/MOM), then run rolling regressions to get exposures. 
mba.tuck.dartmouth.edu

Alt: WRDS mirror if you have access. 
WRDS

3) Fundamentals & quality ratios

What: valuation (PE, PB, EV/EBITDA), margins, growth, leverage, accruals, payout—useful for quality/low-risk tilts and constraint sets.

Where (yf): Ticker.get_income_stmt()/balance_sheet/cashflow/financials + trailing/quarterly/ttm helpers. 
ranaroussi.github.io

Alt: Nasdaq Data Link (Sharadar SF1) for point-in-time standardized fundamentals. 
Nasdaq Data Link
Nasdaq Data Docs

4) Analyst expectations & revisions

What: EPS/revenue estimates, surprise history, revisions, price targets; strong signals for short-horizon BO objectives.

Where (yf): get_earnings_estimate, get_revenue_estimate, get_eps_revisions, analyst_price_targets, get_earnings_history. 
ranaroussi.github.io

5) Events calendar

What: earnings dates, ex-dividends, splits; use to manage blackout windows or add event-aware penalties in the acquisition function.

Where (yf): get_earnings_dates() and calendar. 
ranaroussi.github.io
+1

6) Dividends & buybacks

What: dividend yield/growth, payout stability; shares-outstanding trends to proxy buybacks.

Where (yf): dividends, shares/get_shares_full, actions. 
ranaroussi.github.io

7) Options-implied metrics

What: 30-day IV, IV term structure/skew, put-call ratios—useful as regime/state variables and for downside-aware objectives (e.g., CVaR).

Where (yf): option_chain() for quotes; compute IV/term structure yourself. 
ranaroussi.github.io

Benchmarks: VIX as a market-wide implied vol proxy (also downloadable as ^VIX). Methodology: Cboe docs. 
CBOE
+1

8) Liquidity & microstructure

What: ADV, turnover, Amihud illiquidity, bid-ask/effective spread; helpful for turnover and transaction-cost penalties.

Where (yf): volume/turnover from history() (compute features). 
ranaroussi.github.io

Top-of-book spread/quotes: Polygon.io NBBO quotes; IEX Cloud also exposes real-time/historical quotes. 
Polygon
Postman
PublicAPI

9) Short interest & stock-loan/borrow costs

What: short interest %, days-to-cover, borrow fee/utilization; useful for squeeze/overcrowding risk and borrow-cost-aware objectives.

Where (official): FINRA bi-monthly short interest files + metadata/API guidance. 
FINRA
+2
FINRA
+2

Aggregated: Nasdaq short interest pages/API via Data Link; Polygon short-interest endpoint (mirrors FINRA). 
Nasdaq
Nasdaq Data Docs
Polygon

Borrow fees: Interactive Brokers SLB Rates (GUI/API). 
IBKR Guides
Interactive Brokers

10) Insider & institutional flows

What: insider transactions/holdings; institutional holders; changes can be priors or constraints.

Where (yf): insider_transactions, insider_purchases, insider_roster_holders, institutional_holders. 
ranaroussi.github.io

Primary source: SEC EDGAR APIs (submissions & XBRL). 
SEC
+1

11) ESG & controversies

What: aggregate ESG scores or governance red flags to build “soft” constraints or additional objectives.

Where (yf): sustainability (when available). 
ranaroussi.github.io

Alt: commercial providers (MSCI/Sustainalytics/Refinitiv) if you need coverage/consistency.

12) Macro & rates (for risk-free, constraints, & regimes)

What: risk-free (1M–10Y UST), term spreads, credit spreads, inflation, unemployment, PMI; use as context features or regime switches.

Where: FRED API (and fredapi Python wrapper). 
FRED
GitHub

In [9]:
import pandas as pd
import yfinance as yf
import warnings

warnings.filterwarnings("ignore")
pd.options.display.float_format = '{:.4%}'.format

# Date range
start = '2023-08-18'
end = '2025-08-19'

# Tickers of assets 
assets = ['AVEM', 'XMMO', 'ESGD', 'BYLD', 'ISCF', 'HYEM', 'VWOB', 'VNQI', 'VNQ', 'SNPE'] # , 'AGEPX', 'SWSSX', 'SWVXX' mutual funds 
assets.sort()
raw_data = yf.download(assets, start = start, end = end, auto_adjust=False, interval="1wk")
data = raw_data .loc[:,('Adj Close', slice(None))]
data.columns = assets
data

[*********************100%***********************]  10 of 10 completed


,AVEM,BYLD,ESGD,HYEM,ISCF,SNPE,VNQ,VNQI,VWOB,XMMO
Date,,,,,,,,,,
2023-08-14,4949.4446%,1944.5387%,6591.2033%,1566.7088%,2778.7479%,3898.5134%,7419.4946%,3603.8887%,5345.5383%,7986.3586%
2023-08-21,5029.3350%,1953.0951%,6630.6946%,1569.3445%,2785.1784%,3936.5524%,7465.6120%,3638.6471%,5397.7547%,8024.8962%
2023-08-28,5137.4222%,1963.0020%,6757.6302%,1580.7679%,2847.6427%,4043.8416%,7611.3464%,3709.9930%,5404.8351%,8228.4584%
2023-09-04,5051.8925%,1951.2938%,6661.7233%,1588.7106%,2787.9339%,3990.1978%,7524.6445%,3688.9553%,5407.1217%,8003.1563%
2023-09-11,5098.8869%,1956.6713%,6751.0483%,1592.2445%,2799.8758%,3970.6902%,7545.8572%,3681.6376%,5388.4464%,8024.8962%
...,...,...,...,...,...,...,...,...,...,...
2025-07-21,7052.9999%,2243.9558%,9037.9997%,1977.4057%,3977.9999%,5709.0000%,9173.0003%,4634.0000%,6501.4648%,13211.9995%
2025-07-28,6854.0001%,2256.4001%,8705.0003%,1973.4291%,3871.9002%,5588.0001%,8891.9998%,4550.0000%,6493.5120%,12883.0002%
2025-08-04,7055.0003%,2249.9290%,8997.0001%,1988.9999%,3988.9999%,5738.9999%,8905.9998%,4688.9999%,6576.9997%,13013.0005%


In [10]:
# Calculating returns

Y = data[assets].pct_change().dropna()

display(Y.head())

,AVEM,BYLD,ESGD,HYEM,ISCF,SNPE,VNQ,VNQI,VWOB,XMMO
Date,,,,,,,,,,
2023-08-21,1.6141%,0.4400%,0.5992%,0.1682%,0.2314%,0.9757%,0.6216%,0.9645%,0.9768%,0.4825%
2023-08-28,2.1491%,0.5072%,1.9144%,0.7279%,2.2427%,2.7255%,1.9521%,1.9608%,0.1312%,2.5366%
2023-09-04,-1.6648%,-0.5964%,-1.4192%,0.5025%,-2.0968%,-1.3266%,-1.1391%,-0.5671%,0.0423%,-2.7381%
2023-09-11,0.9302%,0.2756%,1.3409%,0.2224%,0.4283%,-0.4889%,0.2819%,-0.1984%,-0.3454%,0.2716%
2023-09-18,-1.5484%,-0.4067%,-2.1309%,-0.3330%,-1.8045%,-2.9105%,-5.3416%,-1.4410%,-0.7592%,-2.5366%


In [ ]:
# get etf categories 
context_vars = {}

for asset in assets:
    ticker = yf.Ticker(asset)
    info = ticker.info
    category = info.get('category', 'Unknown')
    categories[asset] = category
categories

{'AVEM': 'Diversified Emerging Mkts',
 'BYLD': 'Multisector Bond',
 'ESGD': 'Foreign Large Blend',
 'HYEM': 'Emerging Markets Bond',
 'ISCF': 'Foreign Small/Mid Blend',
 'SNPE': 'Large Blend',
 'VNQ': 'Real Estate',
 'VNQI': 'Global Real Estate',
 'VWOB': 'Emerging Markets Bond',
 'XMMO': 'Mid-Cap Blend'}

In [14]:
ticker = yf.Ticker('ESGD')
info = ticker.info
info['category']

'Foreign Large Blend'

In [22]:
etf = yf.Ticker("ESGD")

fd = etf.funds_data              # or: etf.get_funds_data()
fd.sector_weightings             # -> dict of sector -> weight (0–1)
fd.top_holdings                  # -> DataFrame of top holdings (symbol, name, weight)
fd.equity_holdings               # -> summary ratios (P/E, P/B, etc.)
fd.asset_classes                 # -> {'stocks': ..., 'bonds': ...}


{'cashPosition': 0.0004,
 'stockPosition': 0.99480003,
 'bondPosition': 0.0,
 'preferredPosition': 0.0,
 'convertiblePosition': 0.0,
 'otherPosition': 0.0047999998}

In [25]:
fd.sector_weightings 

{'realestate': 0.0194,
 'consumer_cyclical': 0.0851,
 'basic_materials': 0.049200002,
 'consumer_defensive': 0.0771,
 'technology': 0.10569999,
 'communication_services': 0.0551,
 'financial_services': 0.24780001,
 'utilities': 0.033099998,
 'industrials': 0.1824,
 'energy': 0.0392,
 'healthcare': 0.1058}